# OmniOpinion: Local Connections Baseline Analysis Across City Scales ($10\text{k}$ to $10\text{M}$)

This notebook loads the consolidated Apache Parquet metrics and sample HDF5 trajectories from `data/results/` and generates publication-quality figures:
1. **Polarization & Distance from Randomness Evolution:** Esteban-Ray $P_{ER}$ (Esteban & Ray, 1994), Sarle's Bimodality $BC$ (Pfister et al., 2013), and Bramson Dispersion Ratio vs. $\mathcal{U}(-1, 1)$ (Bramson et al., 2016) with $95\%$ confidence bands across seeds $0\dots 100$.
2. **Effective Number of Opinion Clusters:** Herfindahl-Hirschman Inverse Participation Ratio $N_{\text{eff}}^{\text{IPR}} = 1/\sum \pi_c^2$ (Laakso & Taagepera, 1979) and Shannon Entropy Diversity $\exp(H)$ (Jost, 2006).
3. **Cross-Dimensional Issue Correlation & Dimensionality Collapse:** Baldassarri & Gelman (2008) pairwise issue correlations ($\rho_{ab}$) and Roy's (1953) Participation Ratio $\text{PR}(\Sigma) \in [1, d]$.
4. **Cross-Scale Comparison ($10\text{k}$, $100\text{k}$, $1\text{M}$, $10\text{M}$):** Final distribution across the 101 seeds.
5. **Micro-Level Attitude Phase Space (HDF5 Snapshot):** Joint 2D density & demographic age-susceptibility breakdown from a representative HDF5 trajectory file.

In [1]:
from pathlib import Path
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import seaborn as sns

sns.set_theme(style="whitegrid", context="paper", font_scale=1.25)
plt.rcParams["figure.dpi"] = 140
plt.rcParams["savefig.dpi"] = 300

PROJECT_ROOT = Path("..").resolve() if Path("..").resolve().name == "janones" else Path(".").resolve()
RESULTS_DIR = PROJECT_ROOT / "data" / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Reading data from: {RESULTS_DIR}")
print(f"Saving figures to: {FIGURES_DIR}")

Reading data from: /Users/rennocosta/janones/data/results
Saving figures to: /Users/rennocosta/janones/figures


In [2]:
# Load consolidated Parquet files (or individual metric shards if downloaded directly)
parquet_files = sorted(RESULTS_DIR.glob("**/*.parquet"))
if not parquet_files:
    print("No Parquet files found in data/results/ yet. Running a quick local 3-seed demo to preview plots...")
    import sys
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))
    import yaml
    from src.engines.simulation_engine import SimulationEngine
    with open(PROJECT_ROOT / "configs" / "local_cities_nomedia.yaml") as f:
        demo_cfg = yaml.safe_load(f)
    demo_cfg["simulation"]["n_agents"] = 2000
    demo_cfg["simulation"]["n_steps"] = 60
    demo_dir = RESULTS_DIR / "demo_small_10k"
    for s in range(3):
        SimulationEngine(config=demo_cfg, base_seed=20261030, task_id=s, output_dir=demo_dir).run()
    parquet_files = sorted(RESULTS_DIR.glob("**/*.parquet"))

dfs = []
for pf in parquet_files:
    if pf.name == "all_cities_metrics.parquet" and len(parquet_files) > 1:
        # Prefer the master consolidated file if present
        dfs = [ pq.read_table(pf).to_pandas() ]
        break
    df_part = pq.read_table(pf).to_pandas()
    if "city_scale" not in df_part.columns:
        if "n_agents" in df_part.columns:
            n_val = int(df_part["n_agents"].iloc[0])
            label_map = {10000: "small_10k", 100000: "medium_100k", 1000000: "large_1m", 10000000: "metropolis_10m"}
            df_part["city_scale"] = label_map.get(n_val, f"N_{n_val}")
        else:
            df_part["city_scale"] = pf.parent.name
    dfs.append(df_part)

df = pd.concat(dfs, ignore_index=True)
print(f"Loaded {len(df):,} rows across scales: {df['city_scale'].unique().tolist()} and {df['task_id'].nunique()} seeds.")
df.head()

Loaded 40,804 rows across scales: ['small_10k', 'medium_100k', 'large_1m', 'metropolis_10m'] and 101 seeds.


,step,esteban_ray,sarle_bimodality,dispersion_ratio_vs_random,effective_clusters_ipr,effective_clusters_entropy,occupied_clusters,mean_issue_correlation,max_issue_correlation,effective_dimensionality,...,mean_dist_to_closest_vehicle,tv_reached,radio_reached,press_reached,facts_count,published_notices,pop_cm_d0,pop_cm_d1,pop_cm_d2,pop_cm_d3
0,0,0.001165,0.556240,1.001487,249.144190,252.483706,1,0.007567,0.016409,3.998357,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,0.001125,0.556281,0.941133,248.364520,252.061973,1,0.012906,0.018136,3.997435,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,0.001101,0.555113,0.886904,245.542182,250.520854,1,0.020456,0.031430,3.993662,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,0.001091,0.552753,0.838122,240.646087,247.919063,1,0.027460,0.043722,3.987587,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,0.001094,0.549341,0.793861,234.031997,244.278112,1,0.033875,0.055157,3.979660,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 1. Evolution of Polarization & Distance from Randomness ($t = 0 \dots T$)

In [3]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.2), sharex=True)

sns.lineplot(
    data=df, x="step", y="esteban_ray", hue="city_scale",
    errorbar=("ci", 95), lw=2.2, ax=axes[0]
)
axes[0].set_title("Esteban-Ray Polarization ($P_{ER}$)")
axes[0].set_xlabel("Simulation Step ($t$)")
axes[0].set_ylabel("$P_{ER}$ (Esteban & Ray, 1994)")

sns.lineplot(
    data=df, x="step", y="dispersion_ratio_vs_random", hue="city_scale",
    errorbar=("ci", 95), lw=2.2, ax=axes[1]
)
axes[1].axhline(1.0, color="black", ls="--", lw=1.2, label="Uniform Random ($t=0$)")
axes[1].set_title("Distance from Randomness (Bramson Dispersion)")
axes[1].set_xlabel("Simulation Step ($t$)")
axes[1].set_ylabel("Variance Ratio vs. $\\mathcal{U}(-1, 1)$ ($1/3$)")

sns.lineplot(
    data=df, x="step", y="sarle_bimodality", hue="city_scale",
    errorbar=("ci", 95), lw=2.2, ax=axes[2]
)
axes[2].axhline(5.0 / 9.0, color="crimson", ls="--", lw=1.2, label="Bimodality Threshold ($5/9$)")
axes[2].set_title("Sarle's Bimodality Coefficient ($BC$)")
axes[2].set_xlabel("Simulation Step ($t$)")
axes[2].set_ylabel("$BC$ (Pfister et al., 2013)")
axes[2].legend(loc="best")

plt.tight_layout()
out_fig1 = FIGURES_DIR / "fig1_polarization_and_random_distance.png"
plt.savefig(out_fig1, bbox_inches="tight")
plt.show()
print(f"Saved: {out_fig1}")

<Figure size 2520x728 with 3 Axes>

Saved: /Users/rennocosta/janones/figures/fig1_polarization_and_random_distance.png


## 2. Evolution of the Effective Number of Clusters ($N_{\text{eff}}^{\text{IPR}}$ and $\exp(H)$)

In [4]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.2), sharex=True)

sns.lineplot(
    data=df, x="step", y="effective_clusters_ipr", hue="city_scale",
    errorbar=("ci", 95), lw=2.2, ax=axes[0]
)
axes[0].set_title("Effective Clusters (IPR: $1 / \\sum \\pi_c^2$)")
axes[0].set_xlabel("Simulation Step ($t$)")
axes[0].set_ylabel("$N_{\\text{eff}}^{\\text{IPR}}$ (Laakso & Taagepera, 1979)")

sns.lineplot(
    data=df, x="step", y="effective_clusters_entropy", hue="city_scale",
    errorbar=("ci", 95), lw=2.2, ax=axes[1]
)
axes[1].set_title("Shannon Entropy Cluster Diversity ($\\exp(H)$)")
axes[1].set_xlabel("Simulation Step ($t$)")
axes[1].set_ylabel("$N_{\\text{eff}}^{H}$ (Jost, 2006)")

sns.lineplot(
    data=df, x="step", y="ideological_segregation", hue="city_scale",
    errorbar=("ci", 95), lw=2.2, ax=axes[2]
)
axes[2].set_title("Local Multiplex Ideological Segregation")
axes[2].set_xlabel("Simulation Step ($t$)")
axes[2].set_ylabel("Echo-Chamber Segregation Index $\\in [0, 1]$")

plt.tight_layout()
out_fig2 = FIGURES_DIR / "fig2_effective_clusters_and_segregation.png"
plt.savefig(out_fig2, bbox_inches="tight")
plt.show()
print(f"Saved: {out_fig2}")

<Figure size 2520x728 with 3 Axes>

Saved: /Users/rennocosta/janones/figures/fig2_effective_clusters_and_segregation.png


## 3. Cross-Dimensional Issue Alignment & Effective Dimensionality (Baldassarri & Gelman, 2008)

In [5]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.2), sharex=True)

sns.lineplot(
    data=df, x="step", y="mean_issue_correlation", hue="city_scale",
    errorbar=("ci", 95), lw=2.2, ax=axes[0]
)
axes[0].set_title("Mean Cross-Dimensional Correlation ($\\langle |\\rho_{ab}| \\rangle$)")
axes[0].set_xlabel("Simulation Step ($t$)")
axes[0].set_ylabel("Mean $|\\rho_{ab}|$ (Baldassarri & Gelman, 2008)")

sns.lineplot(
    data=df, x="step", y="effective_dimensionality", hue="city_scale",
    errorbar=("ci", 95), lw=2.2, ax=axes[1]
)
axes[1].set_title("Effective Attitude Dimensionality ($\\text{PR}(\\Sigma)$)")
axes[1].set_xlabel("Simulation Step ($t$)")
axes[1].set_ylabel("Participation Ratio $\\in [1, d]$")

# Plot evolution of all pairwise dimension correlations for the primary scale
corr_cols = [c for c in df.columns if c.startswith("corr_d")]
primary_scale = df["city_scale"].iloc[0]
df_primary = df[df["city_scale"] == primary_scale]
df_corr_long = df_primary.melt(
    id_vars=["step", "task_id"], value_vars=corr_cols,
    var_name="Dimension Pair", value_name="Pearson r"
)
sns.lineplot(
    data=df_corr_long, x="step", y="Pearson r", hue="Dimension Pair",
    errorbar=("ci", 95), lw=2.0, ax=axes[2]
)
axes[2].axhline(0.0, color="black", ls="--", lw=1.0)
axes[2].set_title(f"Pairwise Issue Correlations ({primary_scale})")
axes[2].set_xlabel("Simulation Step ($t$)")
axes[2].set_ylabel("Pearson Correlation $\\rho_{ab}$")

plt.tight_layout()
out_fig3 = FIGURES_DIR / "fig3_issue_correlations_and_dimensionality.png"
plt.savefig(out_fig3, bbox_inches="tight")
plt.show()
print(f"Saved: {out_fig3}")

<Figure size 2520x728 with 3 Axes>

Saved: /Users/rennocosta/janones/figures/fig3_issue_correlations_and_dimensionality.png


## 4. Seed-to-Seed Variation at Final Step ($T = 100$) Across City Scales

In [6]:
max_step = df["step"].max()
df_final = df[df["step"] == max_step].copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5.0))

sns.boxplot(data=df_final, x="city_scale", y="esteban_ray", ax=axes[0], width=0.45)
sns.stripplot(data=df_final, x="city_scale", y="esteban_ray", color="black", alpha=0.4, ax=axes[0])
axes[0].set_title(f"Final Polarization ($t={max_step}$) Across Seeds")
axes[0].set_xlabel("City Scale")
axes[0].set_ylabel("Esteban-Ray $P_{ER}$")

sns.boxplot(data=df_final, x="city_scale", y="effective_clusters_ipr", ax=axes[1], width=0.45)
sns.stripplot(data=df_final, x="city_scale", y="effective_clusters_ipr", color="black", alpha=0.4, ax=axes[1])
axes[1].set_title(f"Final Effective Clusters ($t={max_step}$)")
axes[1].set_xlabel("City Scale")
axes[1].set_ylabel("$N_{\\text{eff}}^{\\text{IPR}}$")

sns.boxplot(data=df_final, x="city_scale", y="mean_issue_correlation", ax=axes[2], width=0.45)
sns.stripplot(data=df_final, x="city_scale", y="mean_issue_correlation", color="black", alpha=0.4, ax=axes[2])
axes[2].set_title(f"Final Issue Alignment ($t={max_step}$)")
axes[2].set_xlabel("City Scale")
axes[2].set_ylabel("Mean $|\\rho_{ab}|$")

plt.tight_layout()
out_fig4 = FIGURES_DIR / "fig4_seed_variation_across_city_scales.png"
plt.savefig(out_fig4, bbox_inches="tight")
plt.show()
print(f"Saved: {out_fig4}")

<Figure size 2520x700 with 3 Axes>

Saved: /Users/rennocosta/janones/figures/fig4_seed_variation_across_city_scales.png


## 5. Micro-Level Attitude Space & IBGE 2022 Age Structure (from Sample HDF5 Trajectory)

In [7]:
h5_files = sorted(RESULTS_DIR.glob("**/*.h5"))
if h5_files:
    sample_h5 = h5_files[0]
    print(f"Inspecting HDF5 file: {sample_h5}")
    with h5py.File(sample_h5, "r") as h5f:
        ages = h5f["static/age"][:]
        suscept = h5f["static/susceptibility"][:]
        innate = h5f["static/innate_beliefs"][:]
        final_ops = h5f["final_state/opinions"][:]
        corr_final = np.corrcoef(final_ops, rowvar=False)

    fig, axes = plt.subplots(1, 4, figsize=(21, 4.8))

    # Panel 1: Initial (t=0) vs Final (t=T) Opinion Distribution on Dim 0 & Dim 1
    idx_sub = np.random.default_rng(42).choice(len(final_ops), size=min(3000, len(final_ops)), replace=False)
    axes[0].scatter(innate[idx_sub, 0], innate[idx_sub, 1], s=6, alpha=0.25, color="gray", label="Initial ($t=0$)")
    axes[0].scatter(final_ops[idx_sub, 0], final_ops[idx_sub, 1], s=8, alpha=0.55, color="royalblue", label="Final ($t=T$)")
    axes[0].set_xlim(-1.05, 1.05)
    axes[0].set_ylim(-1.05, 1.05)
    axes[0].set_title("Attitude Space ($x_1$ vs $x_2$)")
    axes[0].set_xlabel("Dimension 1 (Economy)")
    axes[0].set_ylabel("Dimension 2 (Moral/Customs)")
    axes[0].legend(loc="upper right")

    # Panel 2: Marginal density evolution across all 4 dimensions at t=T
    for d in range(final_ops.shape[1]):
        sns.kdeplot(final_ops[:, d], ax=axes[1], lw=2.0, label=f"Dim {d+1}")
    axes[1].set_xlim(-1.05, 1.05)
    axes[1].set_title("Final Marginal Opinion Densities")
    axes[1].set_xlabel("Attitude $x_{i,d} \\in [-1, 1]$")
    axes[1].legend()

    # Panel 3: Final 4x4 Cross-Dimensional Correlation Matrix Heatmap
    sns.heatmap(
        corr_final, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1,
        xticklabels=[f"D{i+1}" for i in range(corr_final.shape[0])],
        yticklabels=[f"D{i+1}" for i in range(corr_final.shape[0])],
        ax=axes[2]
    )
    axes[2].set_title("Final Issue Correlation Matrix ($\\rho_{ab}$)")

    # Panel 4: IBGE 2022 Age Pyramid & Calibrated Susceptibility
    axes[3].scatter(ages[idx_sub], suscept[idx_sub], s=8, alpha=0.35, color="teal")
    axes[3].set_title(f"IBGE 2022 Age vs. Susceptibility (Med={np.median(ages):.1f}y)")
    axes[3].set_xlabel("Age (Years)")
    axes[3].set_ylabel("Susceptibility $\\alpha_i$")

    plt.tight_layout()
    out_fig5 = FIGURES_DIR / "fig5_micro_attitude_space_and_correlation_heatmap.png"
    plt.savefig(out_fig5, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out_fig5}")
else:
    print("No sample .h5 file found in data/results/ (only Parquet metrics present).")

Inspecting HDF5 file: /Users/rennocosta/janones/data/results/trajectory_task_0000.h5


<Figure size 2940x672 with 5 Axes>

Saved: /Users/rennocosta/janones/figures/fig5_micro_attitude_space_and_correlation_heatmap.png
